# Fig. 2h–j — Vulnerability index workflow and validation in cochlear hair cells

## Concept (Fig. 2h)

Cells with high biosynthetic / translational / energetic demand (`metabolic_load`) relative to their protein-folding-and-clearance capacity (`proteostasis_capacity`) should be more vulnerable to perturbations. We summarise the imbalance with a single per-cell or per-sample score:

$$
\text{vulnerability\_log2ratio} \;=\; \log_2 \bigl(\text{metabolic\_load} + \varepsilon\bigr) - \log_2 \bigl(\text{proteostasis\_capacity} + \varepsilon\bigr)
$$

where each component is the mean of log-normalised expression of a curated gene panel (see [`_gene_sets.py`](_gene_sets.py)):
- **metabolic_load**: OXPHOS (Nduf\*, Cox\*, Atp5\*) + TCA + ion pumps (Atp1\*, Atp2\*) + Ser/Gly/Gln biosynthesis + ribosomal subunits + splicing factors / DEAD-box helicases.
- **proteostasis_capacity**: HSP70/90 + TRiC chaperonin + 20S / 19S proteasome + ubiquitin conjugating + autophagy core + lysosomal hydrolases + ERAD / UPR.

## Validation strategy (Fig. 2i–j)

Cochlear outer hair cells (OHC) are known to be more vulnerable to noise and age than inner hair cells (IHC). We therefore predict OHC vulnerability > IHC vulnerability and test this in **bulk RNA-seq** (Fig. 2i) plus **three independent scRNA-seq datasets** (Fig. 2j):

| Panel | Dataset | Modality | Reference |
| --- | --- | --- | --- |
| Fig. 2i | GSE153882 | Bulk RNA-seq (RPKM) | Liu et al., aged IHC/OHC |
| Fig. 2j | GSE137299 | scRNA-seq (10x), P28 Atoh1-GFP FACS | Kolla et al. |
| Fig. 2j | GSE274279 (3M + 24M) | snRNA-seq (10x) | Cochlea aging atlas |
| Fig. 2j | (Sun et al.) | scRNA-seq | Mouse Cochlea Protein & Cell, adult (1M + 2M) |

**Statistical test**: Mann-Whitney U (two-sided) on per-cell (sc) or per-sample (bulk) vulnerability scores; rank-biserial r reported as effect size.

## Inputs (paths relative to `DATA_DIR = ..`)
- `GSE153882/GSE153882_Aging_Hair_Cells_RPKM.xlsx` — Liu et al. RPKM matrix
- `GSE137299/aa3f0e34.h5ad` — Kolla et al. AnnData (var_names = Ensembl ID; `gene_symbol` in `.var`)
- `Cochlea_aging_atlas/GSE274279_24M_Cochlea_HC.h5ad` — pre-clustered 24M HCs (made once via leiden + Slc17a8/Slc26a5 marker check; see `_preprocess_cochlea_aging_atlas` helper below)
- `Cochlea_aging_atlas/GSE274279_3M_Cochlea_HC.h5ad` — pre-clustered 3M HCs
- `Sun_etal_Mouse_Cochlea.h5ad` — Sun et al. adata with `CellType`, `age`, `leiden`

## Outputs
- `figures/Fig2i_bulk_GSE153882_9M.pdf` — Liu et al. 9-month bulk
- `figures/Fig2j_Kolla_GSE137299.pdf`
- `figures/Fig2j_CochleaAging_GSE274279_3M.pdf`, `..._24M.pdf`
- `figures/Fig2j_Sun_adult.pdf`
- `figures/Fig2hij_summary_stats.csv` — n, median, MWU p, rank-biserial r per dataset

## Caveats (please flag during reviewer pass)
- **Public scRNA-seq IHC/OHC labels** come from re-clustering each dataset and reading off `Slc17a8` (IHC) vs `Slc26a5` (OHC) marker expression on the leiden UMAP. Cluster numbering is not stable across reruns; the mapping dict in `_preprocess_*` must be re-verified each time the upstream clustering changes.
- **Gene panels are hand-curated**, not derived from a database. They were chosen to maximise generality (well-conserved housekeeping families) and avoid sparsely-expressed regulators. Robustness against an alternative HALLMARK-based panel is checked in [`RNA_vul_robustness_0423.ipynb`](../RNA_vul_robustness_0423.ipynb) (not included in this clean repo).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

from _gene_sets import metabolic_genes, proteostasis_genes

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(4, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

EPS = 1e-6
PALETTE = {'IHC': '#999999', 'OHC': '#555555'}
VUL_CLIP = 15  # drop pathological log2-ratio outliers (sparse cells with both denom ~0 and noisy numerator)

print(f'metabolic_genes panel : {len(metabolic_genes)} genes')
print(f'proteostasis_genes panel: {len(proteostasis_genes)} genes')

## Helper functions

Two scoring helpers — kept distinct on purpose:
- `geneset_mean`: log-normalised raw mean over a panel. Used for bulk (Fig 2i) and public scRNA-seq (Fig 2j).
- `geneset_score_genes`: scanpy's `sc.tl.score_genes` (background-corrected, can be negative). Used in [Fig2kl.ipynb](Fig2kl.ipynb) for the in-house Smart-seq3 VHC data, where higher dynamic range is needed to separate fine subtypes.

The Fig 2i–j (`geneset_mean`) and Fig 2k–l (`score_genes`) implementations are intentionally different: the public-dataset version preserves absolute magnitude on a comparable scale across datasets, while the in-house version corrects for cell-level expression background within a single dataset. Reviewer should be aware of this asymmetry.

In [ ]:
def geneset_mean(adata, genes):
    """Mean log1p-normalised expression of a gene panel, per cell.

    Assumes adata.X is already normalize_total + log1p transformed.
    Genes not in adata.var_names are silently dropped (caller already filtered).
    """
    X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
    idx = adata.var_names.get_indexer(genes)
    return X[:, idx].mean(axis=1)


def compute_vulnerability_sc(adata, layer_counts='counts'):
    """Add metabolic_load / proteostasis_capacity / vulnerability_log2ratio to .obs.

    Pipeline:
      1. Copy counts back to .X.
      2. normalize_total + log1p (10k library size).
      3. Filter panels to genes present in .var_names.
      4. Per-cell mean over each panel.
      5. log2(metabolic) - log2(proteostasis), with eps to handle zero denominator.
    """
    ad_v = adata.copy()
    if layer_counts in ad_v.layers:
        ad_v.X = ad_v.layers[layer_counts].copy()
    sc.pp.normalize_total(ad_v, target_sum=1e4)
    sc.pp.log1p(ad_v)

    met_use = [g for g in metabolic_genes if g in ad_v.var_names]
    pro_use = [g for g in proteostasis_genes if g in ad_v.var_names]
    print(f'  metabolic    used: {len(met_use)}/{len(metabolic_genes)}')
    print(f'  proteostasis used: {len(pro_use)}/{len(proteostasis_genes)}')

    ad_v.obs['metabolic_load'] = geneset_mean(ad_v, met_use)
    ad_v.obs['proteostasis_capacity'] = geneset_mean(ad_v, pro_use)
    ad_v.obs['vulnerability_log2ratio'] = (
        np.log2(ad_v.obs['metabolic_load'] + EPS) -
        np.log2(ad_v.obs['proteostasis_capacity'] + EPS)
    )
    return ad_v


def boxplot_ihc_ohc(plot_df, title, save_path):
    plot_df = plot_df.copy()
    plot_df = plot_df[plot_df['cell_type'].isin(['IHC', 'OHC'])]
    plot_df = plot_df[plot_df['vulnerability_log2ratio'] <= VUL_CLIP]
    plot_df['cell_type'] = pd.Categorical(plot_df['cell_type'], categories=['IHC', 'OHC'], ordered=True)

    fig, ax = plt.subplots(figsize=(4, 5))
    sns.boxplot(
        data=plot_df, x='cell_type', y='vulnerability_log2ratio',
        ax=ax, hue='cell_type', palette=PALETTE,
        boxprops=dict(facecolor='none'), legend=False,
    )
    sns.stripplot(
        data=plot_df, x='cell_type', y='vulnerability_log2ratio',
        ax=ax, hue='cell_type', palette=PALETTE, legend=False,
        size=2, alpha=0.5, jitter=0.2,
    )
    ax.set_xlabel('')
    ax.set_ylabel('Vulnerability (log2 metabolic - proteostasis)')
    ax.set_title(title)

    ihc = plot_df.loc[plot_df['cell_type'] == 'IHC', 'vulnerability_log2ratio'].dropna()
    ohc = plot_df.loc[plot_df['cell_type'] == 'OHC', 'vulnerability_log2ratio'].dropna()
    stat, pval = mannwhitneyu(ihc, ohc, alternative='two-sided')
    n1, n2 = len(ihc), len(ohc)
    rb = 1 - 2 * stat / (n1 * n2) if n1 * n2 > 0 else np.nan

    ymax = plot_df['vulnerability_log2ratio'].max()
    ax.text(0.5, ymax * 1.02, f'p = {pval:.2e}\n(MWU)',
            ha='center', va='bottom', transform=ax.get_xaxis_transform(), fontsize=9)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'IHC_n': n1, 'IHC_median': float(ihc.median()),
        'OHC_n': n2, 'OHC_median': float(ohc.median()),
        'median_diff_OHC_minus_IHC': float(ohc.median() - ihc.median()),
        'MWU_U': float(stat), 'MWU_p': float(pval),
        'rank_biserial_r': float(rb),
    }

---
## Fig 2i — Bulk RNA-seq (Liu et al. GSE153882)

16 samples (4 conditions × 4 replicates): IHC/OHC × 9M / 26M. We focus on the 9-month subset to keep developmental noise low; the same trend (OHC > IHC vulnerability) is preserved at 26M but with smaller effect size because both cell types accumulate age-related stress.

In [ ]:
fn = DATA_DIR / 'GSE153882' / 'GSE153882_Aging_Hair_Cells_RPKM.xlsx'
df = pd.read_excel(fn)
df = df.dropna(subset=['Name']).drop_duplicates(subset=['Name']).set_index('Name')
expr_cols = [c for c in df.columns if c.endswith('RPKM value')]
expr = df[expr_cols]

# parse sample metadata from column names: e.g. '9m_IHCs_1 RPKM value'
meta = pd.DataFrame(index=expr.columns)
idx = meta.index.to_series()
meta['age_m'] = idx.str.extract(r'^(\d+)m')[0].astype(int)
meta['cell_type'] = idx.str.extract(r'm_(IHCs|OHCs)_')[0].str.rstrip('s')  # IHCs -> IHC
meta['rep'] = idx.str.extract(r'_(\d+)\sRPKM')[0].astype(int)

# bulk version of compute_vulnerability_sc: log2(RPKM+1) mean per panel
logexpr = np.log2(expr + 1.0)
met_use = [g for g in metabolic_genes if g in logexpr.index]
pro_use = [g for g in proteostasis_genes if g in logexpr.index]
print(f'metabolic    used: {len(met_use)}/{len(metabolic_genes)}')
print(f'proteostasis used: {len(pro_use)}/{len(proteostasis_genes)}')

meta['metabolic_load'] = logexpr.loc[met_use].mean(axis=0).values
meta['proteostasis_capacity'] = logexpr.loc[pro_use].mean(axis=0).values
meta['vulnerability_log2ratio'] = meta['metabolic_load'] - meta['proteostasis_capacity']
meta

In [ ]:
# 9-month subset (focus age)
df_9m = meta.query('age_m == 9').copy()

fig, ax = plt.subplots(figsize=(3, 4))
sns.boxplot(
    data=df_9m, x='cell_type', y='vulnerability_log2ratio',
    width=0.5, showfliers=True, ax=ax, hue='cell_type',
    palette=PALETTE, boxprops=dict(facecolor='none'), legend=False,
)
sns.stripplot(
    data=df_9m, x='cell_type', y='vulnerability_log2ratio',
    ax=ax, color='black', size=5, jitter=0.15, alpha=0.8,
)

ihc = df_9m.loc[df_9m['cell_type'] == 'IHC', 'vulnerability_log2ratio']
ohc = df_9m.loc[df_9m['cell_type'] == 'OHC', 'vulnerability_log2ratio']
stat, pval = mannwhitneyu(ihc, ohc, alternative='two-sided')
ax.text(0.5, df_9m['vulnerability_log2ratio'].max() * 1.02, f'p = {pval:.2e}\n(MWU)',
        ha='center', va='bottom', transform=ax.get_xaxis_transform(), fontsize=9)

ax.set_xlabel('')
ax.set_ylabel('Vulnerability (log2 metabolic - proteostasis)')
ax.set_title('9-month hair cells (bulk RNA-seq, GSE153882)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'Fig2i_bulk_GSE153882_9M.pdf', dpi=300, bbox_inches='tight')
plt.show()

summary_rows = [{
    'panel': 'Fig 2i', 'dataset': 'GSE153882 (bulk, 9M)',
    'IHC_n': len(ihc), 'IHC_median': float(ihc.median()),
    'OHC_n': len(ohc), 'OHC_median': float(ohc.median()),
    'median_diff_OHC_minus_IHC': float(ohc.median() - ihc.median()),
    'MWU_U': float(stat), 'MWU_p': float(pval), 'rank_biserial_r': np.nan,
}]

---
## Fig 2j — scRNA-seq dataset 1: Kolla et al. (GSE137299)

P28 mouse cochlea, Atoh1-GFP+ FACS, 10x Genomics. Cell-type labels (`cell_type` = IHC / OHC / Type II / SC / ...) are already in `.obs` from the deposited h5ad. We just need to switch `var_names` from Ensembl ID to gene symbol.

In [ ]:
adata_kolla = sc.read_h5ad(DATA_DIR / 'GSE137299' / 'aa3f0e34.h5ad')
adata_kolla.var['ensembl_ID'] = adata_kolla.var.index
adata_kolla.var_names = adata_kolla.var['gene_symbol'].astype(str).values
adata_kolla.var_names_make_unique()
adata_kolla.layers['counts'] = adata_kolla.X.copy()

print('Cell types in Kolla et al.:')
print(adata_kolla.obs['cell_type'].value_counts())

In [ ]:
adata_kolla_v = compute_vulnerability_sc(adata_kolla)
stats = boxplot_ihc_ohc(
    adata_kolla_v.obs,
    title='Kolla et al. 2020 (P28, Atoh1-GFP FACS)',
    save_path=FIG_DIR / 'Fig2j_Kolla_GSE137299.pdf',
)
summary_rows.append({'panel': 'Fig 2j', 'dataset': 'GSE137299 (Kolla P28)', **stats})

---
## Fig 2j — scRNA-seq dataset 2: Cochlea aging atlas (GSE274279, 3M + 24M)

snRNA-seq. Raw data is the 10x triplet (`matrix.mtx`, `features.tsv`, `barcodes.tsv`) per sample; we cluster with default scanpy parameters and assign IHC/OHC by marker expression (`Slc17a8` = IHC, `Slc26a5` = OHC). After this assignment we save a small h5ad so the IHC/OHC labels are persistent.

**Caveat**: leiden cluster numbers change between runs / scanpy versions. The mapping in `_preprocess_cochlea_aging_atlas` was derived on a specific run — re-verify by `sc.pl.umap(adata, color=['leiden', 'Slc17a8', 'Slc26a5'])` before trusting the labels.

In [ ]:
from scipy.io import mmread

def _load_10x_triplet(mtx_fp, feat_fp, bar_fp):
    X = mmread(mtx_fp).T.tocsr()  # genes x cells -> cells x genes
    features = pd.read_csv(feat_fp, sep='\t', header=None,
                           names=['gene_id', 'gene_symbol', 'feature_type'])
    barcodes = pd.read_csv(bar_fp, sep='\t', header=None, names=['barcode'])
    adata = sc.AnnData(X=X)
    adata.var_names = features['gene_symbol'].values
    adata.var['gene_id'] = features['gene_id'].values
    adata.obs_names = barcodes['barcode'].values
    adata.var_names_make_unique()
    return adata


def _preprocess_cochlea_aging_atlas(sample, cluster_to_celltype, hvg_top=1500, leiden_res=0.8):
    """Load mtx triplet, cluster, assign IHC/OHC by leiden cluster number.

    Args
    ----
    sample: e.g. 'GSM8446833_3M_Cochlea'
    cluster_to_celltype: dict mapping leiden id (str) -> 'IHC' or 'OHC'
    """
    base = DATA_DIR / 'Cochlea_aging_atlas'
    adata = _load_10x_triplet(
        base / f'{sample}_matrix.mtx',
        base / f'{sample}_features.tsv',
        base / f'{sample}_barcodes.tsv',
    )
    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=hvg_top, flavor='seurat_v3', layer='counts')
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50)
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=leiden_res, key_added='leiden')

    # diagnostic: marker UMAP for cluster ID verification
    sc.pl.umap(adata, color=['leiden', 'Slc17a8', 'Slc26a5'], legend_loc='on data')

    adata.obs['cell_type'] = adata.obs['leiden'].astype(str).map(cluster_to_celltype).fillna('other')
    adata_HC = adata[adata.obs['cell_type'].isin(['IHC', 'OHC'])].copy()
    print(f'\n{sample}: HC subset = {adata_HC.n_obs} cells')
    print(adata_HC.obs['cell_type'].value_counts())
    return adata_HC

In [ ]:
# 3M sample — cluster-to-cell-type mapping derived from the marker UMAP above.
# RE-VERIFY this dict if you re-cluster the dataset.
adata_3M = _preprocess_cochlea_aging_atlas(
    'GSM8446833_3M_Cochlea',
    cluster_to_celltype={'4': 'IHC', '6': 'OHC'},
)
adata_3M_v = compute_vulnerability_sc(adata_3M)
stats = boxplot_ihc_ohc(
    adata_3M_v.obs,
    title='Cochlea aging atlas — 3M (GSE274279)',
    save_path=FIG_DIR / 'Fig2j_CochleaAging_GSE274279_3M.pdf',
)
summary_rows.append({'panel': 'Fig 2j', 'dataset': 'GSE274279 (3M)', **stats})

In [ ]:
# 24M sample. The original analysis used `sc.tl.leiden(restrict_to=['leiden', ['9']])`
# to sub-cluster HC cluster 9 into IHC ('9,1') and OHC ('9,2') because the parent cluster
# was a mixed HC pool. We replicate that here.
base = DATA_DIR / 'Cochlea_aging_atlas'
sample = 'GSM8446835_24M_Cochlea'
adata_24M_full = _load_10x_triplet(
    base / f'{sample}_matrix.mtx',
    base / f'{sample}_features.tsv',
    base / f'{sample}_barcodes.tsv',
)
adata_24M_full.layers['counts'] = adata_24M_full.X.copy()
sc.pp.normalize_total(adata_24M_full, target_sum=1e4)
sc.pp.log1p(adata_24M_full)
sc.pp.highly_variable_genes(adata_24M_full, n_top_genes=1500, flavor='seurat_v3', layer='counts')
sc.pp.scale(adata_24M_full, max_value=10)
sc.tl.pca(adata_24M_full, n_comps=50)
sc.pp.neighbors(adata_24M_full, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_24M_full)
sc.tl.leiden(adata_24M_full, resolution=0.8)
# sub-cluster the HC pool (cluster '9' on the original run)
sc.tl.leiden(adata_24M_full, resolution=0.8, restrict_to=['leiden', ['9']], key_added='leiden_HC')
sc.pl.umap(adata_24M_full, color=['leiden', 'leiden_HC', 'Slc17a8', 'Slc26a5'], legend_loc='on data')

# RE-VERIFY mapping below if leiden numbering shifts
adata_24M_full.obs['cell_type'] = adata_24M_full.obs['leiden_HC'].astype(str).map(
    {'9,1': 'IHC', '9,2': 'OHC'}
).fillna('other')
adata_24M = adata_24M_full[adata_24M_full.obs['cell_type'].isin(['IHC', 'OHC'])].copy()
print(f'\n24M HC subset = {adata_24M.n_obs}')
print(adata_24M.obs['cell_type'].value_counts())

adata_24M_v = compute_vulnerability_sc(adata_24M)
stats = boxplot_ihc_ohc(
    adata_24M_v.obs,
    title='Cochlea aging atlas — 24M (GSE274279)',
    save_path=FIG_DIR / 'Fig2j_CochleaAging_GSE274279_24M.pdf',
)
summary_rows.append({'panel': 'Fig 2j', 'dataset': 'GSE274279 (24M)', **stats})

---
## Fig 2j — scRNA-seq dataset 3: Sun et al. Protein & Cell (adult, 1M + 2M)

The deposited adata contains all ages (1M / 2M / 5M). 5M is dropped because age-related expression shifts have begun and could confound the IHC/OHC vulnerability comparison. After subsetting to 1M+2M and to `CellType == 'HC'`, we re-cluster the HC pool and assign IHC/OHC by leiden cluster ID + marker UMAP.

In [ ]:
adata_sun = sc.read_h5ad(DATA_DIR / 'Sun_etal_Mouse_Cochlea.h5ad')
print('HC by age:')
print(adata_sun.obs.loc[adata_sun.obs['CellType'] == 'HC', 'age'].value_counts().sort_index())

adult_ages = ['1M', '2M']
adata_sun_HC = adata_sun[
    (adata_sun.obs['CellType'] == 'HC') & (adata_sun.obs['age'].isin(adult_ages))
].copy()
adata_sun_HC.layers['counts'] = adata_sun_HC.X.copy()
print(f'\nAdult HC subset: {adata_sun_HC.n_obs} cells')

# re-cluster within HC pool
sc.pp.normalize_total(adata_sun_HC, target_sum=1e4)
sc.pp.log1p(adata_sun_HC)
sc.pp.highly_variable_genes(adata_sun_HC, n_top_genes=150, flavor='seurat_v3', layer='counts')
sc.pp.scale(adata_sun_HC, max_value=10)
adata_sun_HC.X = np.nan_to_num(adata_sun_HC.X, nan=0.0)
sc.tl.pca(adata_sun_HC, n_comps=20)
sc.pp.neighbors(adata_sun_HC, n_neighbors=40, n_pcs=20)
sc.tl.umap(adata_sun_HC)
sc.tl.leiden(adata_sun_HC, resolution=1.0)

sc.pl.umap(
    adata_sun_HC,
    color=['leiden', 'age',
           'Pou4f3', 'Myo7a', 'Myo6',  # pan-HC
           'Slc17a8', 'Otof', 'Fgf8',  # IHC
           'Slc26a5', 'Ocm', 'Ikzf2'], # OHC
    ncols=3,
)

In [ ]:
# Manual mapping derived from the marker UMAP above — RE-VERIFY each rerun.
sun_cluster_to_celltype = {
    '0': 'NA',
    '1': 'IHC',
    '2': 'IHC',
    '3': 'NA',
    '4': 'OHC',
}
adata_sun_HC.obs['cell_type'] = adata_sun_HC.obs['leiden'].astype(str).map(sun_cluster_to_celltype)
adata_sun_HC = adata_sun_HC[adata_sun_HC.obs['cell_type'].isin(['IHC', 'OHC'])].copy()
print(adata_sun_HC.obs['cell_type'].value_counts())

adata_sun_v = compute_vulnerability_sc(adata_sun_HC)
stats = boxplot_ihc_ohc(
    adata_sun_v.obs,
    title='Sun et al. — adult (1M + 2M)',
    save_path=FIG_DIR / 'Fig2j_Sun_adult.pdf',
)
summary_rows.append({'panel': 'Fig 2j', 'dataset': 'Sun adult (1M+2M)', **stats})

---
## Summary statistics across all panels

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(FIG_DIR / 'Fig2hij_summary_stats.csv', index=False)
summary_df